In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader

In [2]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,)) # mean, std
])

In [3]:
train_dataset = torchvision.datasets.MNIST(
    root="./data", train=True, download=True, transform=transform)
test_dataset = torchvision.datasets.MNIST(
    root="./data", train=True, download=True, transform=transform)

In [4]:
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=1000, shuffle=False)

In [5]:
class MNISTClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()
        self.layers = nn.Sequential(
            nn.Linear(784, 128),
            nn.ReLU(),
            nn.Linear(128, 10)
        )
    
    def forward(self, x):
        x = self.flatten(x)
        x = self.layers(x)
        return x

In [6]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using {device}")

model = MNISTClassifier().to(device)

loss_function = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

Using cpu


In [7]:
def train_epoch(model, train_loader, loss_function, optimizer, device):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for batch_idx, (data, targets) in enumerate(train_loader):
        data, targets = data.to(device), targets.to(device)

        optimizer.zero_grad()
        output = model(data)
        loss = loss_function(output, targets)
        loss.backward()
        optimizer.step()

        # Track progress
        running_loss += loss.item()
        _, predicted = output.max(1)
        total += targets.size(0)
        correct += (predicted == targets).sum().item()

        if batch_idx % 100 == 0 and batch_idx > 0:
            avg_loss = running_loss / 100
            accuracy = 100 * correct / total
            print(f" Progress: [{batch_idx * 64} / {60000}]\n"
                f"Loss: {avg_loss:.3f} | Accuracy: {accuracy:.1f}%")

            running_loss = 0.0

In [10]:
def evaluate(model, test_loader, device):
    model.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for inputs, targets in test_loader:
            inputs, targets = inputs.to(device), targets.to(device)
            outputs = model(inputs)
            _, predicted = outputs.max(1)
            correct += (predicted == targets).sum().item()
            total += targets.size(0)
    
    return 100 * correct / total

In [11]:
# Traning Loop
num_epochs = 10
for epoch in range(num_epochs):
    print(f"\nEpoch {epoch + 1}")
    train_epoch(model, train_loader, loss_function, optimizer, device)
    accuracy = evaluate(model, test_loader, device)
    print(f"Accuracy: {accuracy:.1f}")


Epoch 1
 Progress: [6400 / 60000]
Loss: 0.118 | Accuracy: 96.4%
 Progress: [12800 / 60000]
Loss: 0.118 | Accuracy: 96.4%
 Progress: [19200 / 60000]
Loss: 0.122 | Accuracy: 96.4%
 Progress: [25600 / 60000]
Loss: 0.113 | Accuracy: 96.5%
 Progress: [32000 / 60000]
Loss: 0.120 | Accuracy: 96.5%
 Progress: [38400 / 60000]
Loss: 0.103 | Accuracy: 96.6%
 Progress: [44800 / 60000]
Loss: 0.097 | Accuracy: 96.7%
 Progress: [51200 / 60000]
Loss: 0.122 | Accuracy: 96.6%
 Progress: [57600 / 60000]
Loss: 0.091 | Accuracy: 96.7%
Accuracy: 97.8

Epoch 2
 Progress: [6400 / 60000]
Loss: 0.083 | Accuracy: 97.5%
 Progress: [12800 / 60000]
Loss: 0.075 | Accuracy: 97.6%
 Progress: [19200 / 60000]
Loss: 0.075 | Accuracy: 97.7%
 Progress: [25600 / 60000]
Loss: 0.078 | Accuracy: 97.6%
 Progress: [32000 / 60000]
Loss: 0.066 | Accuracy: 97.7%
 Progress: [38400 / 60000]
Loss: 0.081 | Accuracy: 97.7%
 Progress: [44800 / 60000]
Loss: 0.087 | Accuracy: 97.6%
 Progress: [51200 / 60000]
Loss: 0.089 | Accuracy: 97.5%
